In [30]:
# ==================================================
# CELL 0: Global Setup & Imports
# ==================================================

# 1. INSTALL DEPENDENCIES (Jalankan sekali)
!pip install -q transformers albumentations timm wandb opencv-python

# 3. STANDARD LIBRARIES
import json
import math
import random
import zipfile
import warnings
warnings.filterwarnings('ignore')

# 4. DATA & IMAGE PROCESSING
import numpy as np
import pandas as pd
import cv2
from PIL import Image

# 5. PYTORCH CORE
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, BatchSampler, ConcatDataset

# 6. AUGMENTATION
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 7. HUGGINGFACE TRANSFORMERS
from transformers import Mask2FormerForUniversalSegmentation
from transformers import SegformerForSemanticSegmentation

# 8. UTILITIES & TRACKING
from tqdm.auto import tqdm
try:
    import wandb
    WANDB_AVAILABLE = True
    print("[OK] WANDB is available.")
except ImportError:
    WANDB_AVAILABLE = False
    print("[WARN] WANDB is not available.")

print("[OK] All libraries successfully imported!")

[OK] WANDB is available.
[OK] All libraries successfully imported!


In [31]:
# ==================================================
# CELL 1: Setup & Mount Drive
# ==================================================
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Colab: {IN_COLAB}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab: True


In [32]:
# ==================================================
# CELL 2: Extract Dataset & Set Seed
# ==================================================
import os
import zipfile
import random
import numpy as np
import torch

ZIP_PATH = '/content/drive/MyDrive/flood_segmentation/dolanan-data-nexus-2026.zip'
EXTRACT_PATH = '/content/flood_project'

if not os.path.exists(f'{EXTRACT_PATH}/Dataset_Nexus_DolananData'):
    os.makedirs(EXTRACT_PATH, exist_ok=True)
    print("Extracting (5-10 menit)...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print("Done")
else:
    print("Already extracted")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

Already extracted


In [33]:
# ==================================================
# CELL 3: Base Paths & Folder Mapping
# ==================================================
possible_paths = [
    "/content/flood_project/Dataset_Nexus_DolananData",
    "/content/drive/MyDrive/flood_segmentation/Dataset_Nexus_DolananData",
]
BASE_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        BASE_PATH = p
        break
if BASE_PATH is None:
    raise FileNotFoundError("Dataset not found")
print(f"BASE_PATH: {BASE_PATH}")

def find_mask_dir(base):
    for item in os.listdir(base):
        if item.startswith('train-'):
            train_path = os.path.join(base, item)
            for sub in os.listdir(train_path):
                train_dir = os.path.join(train_path, sub)
                if os.path.isdir(train_dir):
                    for name in ["masks", "mask"]:
                        if os.path.isdir(os.path.join(train_dir, name)):
                            return name
    raise FileNotFoundError("mask/masks folder not found")

MASK_DIR = find_mask_dir(BASE_PATH)

train_folder = val_folder = test_folder = None
for item in os.listdir(BASE_PATH):
    item_path = os.path.join(BASE_PATH, item)
    if os.path.isdir(item_path):
        if item.startswith('train-'): train_folder = item
        elif item.startswith('validation-'): val_folder = item
        elif item.startswith('test-'): test_folder = item

TRAIN_IMG  = os.path.join(BASE_PATH, train_folder, "train", "img")
TRAIN_MASK = os.path.join(BASE_PATH, train_folder, "train", MASK_DIR)
VAL_IMG    = os.path.join(BASE_PATH, val_folder, "validation", "img")
VAL_MASK   = os.path.join(BASE_PATH, val_folder, "validation", MASK_DIR)
test_mid   = os.listdir(os.path.join(BASE_PATH, test_folder))[0]
TEST_IMG   = os.path.join(BASE_PATH, test_folder, test_mid, "test", "img")

print(f"TRAIN_IMG: {TRAIN_IMG}\nVAL_IMG: {VAL_IMG}\nTEST_IMG: {TEST_IMG}")

BASE_PATH: /content/flood_project/Dataset_Nexus_DolananData
TRAIN_IMG: /content/flood_project/Dataset_Nexus_DolananData/train-20260429T073729Z-3-001/train/img
VAL_IMG: /content/flood_project/Dataset_Nexus_DolananData/validation-20260429T073727Z-3-001/validation/img
TEST_IMG: /content/flood_project/Dataset_Nexus_DolananData/test-20260429T073643Z-3-001/test-20260429T073643Z-3-001/test/img


In [34]:
# ==================================================
# CELL 4: Constants, Model Config & Checkpoint Paths
# ==================================================
CLASS_NAMES = {
    0: "background", 1: "building flooded", 2: "building non-flooded",
    3: "grass", 4: "pool", 5: "road flooded",
    6: "road non-flooded", 7: "tree", 8: "vehicle", 9: "water"
}
NUM_CLASSES = 10
EMPTY_CLASSES = {2, 6}
RARE_CLASSES = {4, 8, 9}

CLASS_WEIGHTS = torch.tensor([0.3, 1.0, 0.0, 0.5, 3.0, 0.8, 0.0, 1.2, 15.0, 20.0], dtype=torch.float32)
CLASS_WEIGHTS_ENHANCED = torch.tensor([0.3, 1.0, 0.0, 0.5, 3.0, 0.8, 0.0, 1.2, 25.0, 35.0], dtype=torch.float32)

# Swin-Base pretrained with ImageNet stats
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

MODEL_NAME     = "facebook/mask2former-swin-base-ade-semantic"
SEGFORMER_NAME = "nvidia/mit-b4"

TRAIN_CONFIG = {
    'num_epochs': 35,
    'lr_backbone': 3e-5,
    'lr_head': 1e-4,
    'weight_decay': 0.01,
    'grad_clip': 1.0,
    'save_dir': '/content/drive/MyDrive/flood_segmentation/checkpoints_mask2former',
    'early_stop_patience': 7,
    'accumulation_steps': 8,
    'lovasz_weight': 0.75,
    'rare_ratio': 0.40,
    'water_ratio': 0.15,
    'batch_size': 2,
    'poly_power': 0.9,
    'warmup_steps': 500,
}

TRAIN_CONFIG_FINETUNE = TRAIN_CONFIG.copy()
TRAIN_CONFIG_FINETUNE.update({
    'num_epochs': 15,
    'lr_backbone': 1e-5,
    'lr_head': 5e-5,
    'accumulation_steps': 8,
    'warmup_steps': 200,
})

os.makedirs(TRAIN_CONFIG['save_dir'], exist_ok=True)
PSEUDO_MASK_DIR = os.path.join(EXTRACT_PATH, 'pseudo_masks')
os.makedirs(PSEUDO_MASK_DIR, exist_ok=True)

# Konstanta jalur Checkpoint
CKPT_M2F          = os.path.join(TRAIN_CONFIG['save_dir'], 'best_mask2former.pth')
CKPT_M2F_FINETUNE = os.path.join(TRAIN_CONFIG['save_dir'], 'best_m2f_finetune.pth')
CKPT_M2F_WEIGHTED = os.path.join(TRAIN_CONFIG['save_dir'], 'best_m2f_weighted.pth')
CKPT_M2F_PSEUDO   = os.path.join(TRAIN_CONFIG['save_dir'], 'best_m2f_pseudo.pth')
CKPT_SEG          = os.path.join(TRAIN_CONFIG['save_dir'], 'best_segformer.pth')

print(f"MASK_DIR: {MASK_DIR}\nModel: {MODEL_NAME}\nConfig ready")


MASK_DIR: masks
Model: facebook/mask2former-swin-base-ade-semantic
Config ready


In [35]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==================================================
# CELL 5: Augmentations
# ==================================================

train_transform = A.Compose([
    # [FIX] Menambahkan mask_value=255 agar padding tidak dihitung sebagai background oleh model
    A.PadIfNeeded(min_height=512, min_width=512, border_mode=0, value=0, mask_value=255),
    A.RandomCrop(512, 512),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.2),
    # [FIX] Menambahkan mask_value=255 untuk rotasi agar rotasi background terabaikan
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=0, value=0, mask_value=255),
    A.GridDistortion(p=0.3),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                    min_holes=1, min_height=8, min_width=8, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# FIX #8: val_transform pakai resize ke resolusi asli (640x480)
# agar metric validasi konsisten dengan submission
val_transform = A.Compose([
    A.Resize(480, 640),   # height=480, width=640 → match original image size
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

print("Augmentations ready")
print("[FIX #8] val_transform sekarang resize ke 640x480 (bukan 512x512)")
print("         Konsisten dengan submission output size")

Augmentations ready
[FIX #8] val_transform sekarang resize ke 640x480 (bukan 512x512)
         Konsisten dengan submission output size


In [36]:
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from tqdm.autonotebook import tqdm

# ==================================================
# CELL 6: Dataset Classes
# ==================================================

class FloodSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, image_ids=None):
        self.img_dir  = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        all_ids   = sorted([os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith('.jpg')])
        self.ids  = image_ids if image_ids is not None else all_ids

        self.rare_classes_in_image = {}
        self.water_in_image        = set()
        for img_id in tqdm(self.ids, desc="Indexing"):
            mask_path = os.path.join(mask_dir, f"{img_id}.png")
            if os.path.exists(mask_path):
                mask   = np.array(Image.open(mask_path))
                mask[(mask >= NUM_CLASSES) & (mask != 255)] = 0
                unique = np.unique(mask)
                self.rare_classes_in_image[img_id] = set(unique) & RARE_CLASSES
                if 9 in unique:
                    self.water_in_image.add(img_id)
            else:
                self.rare_classes_in_image[img_id] = set()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id   = self.ids[idx]
        img      = np.array(Image.open(os.path.join(self.img_dir,  f"{img_id}.jpg")).convert('RGB'))
        mask     = np.array(Image.open(os.path.join(self.mask_dir, f"{img_id}.png")))
        mask[(mask >= NUM_CLASSES) & (mask != 255)] = 0
        if self.transform:
            transformed = self.transform(image=img, mask=mask)
            img  = transformed['image']
            mask = transformed['mask'].long()
        return img, mask, img_id

    def get_rare_indices(self):
        return [i for i, img_id in enumerate(self.ids)
                if len(self.rare_classes_in_image.get(img_id, set())) > 0]

    def get_water_indices(self):
        return [i for i, img_id in enumerate(self.ids) if img_id in self.water_in_image]

    def get_normal_indices(self):
        rare = set(self.get_rare_indices())
        return [i for i in range(len(self.ids)) if i not in rare]


class FloodTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir   = img_dir
        self.transform = transform
        self.ids       = sorted([os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith('.jpg')])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img    = np.array(Image.open(os.path.join(self.img_dir, f"{img_id}.jpg")).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, img_id


class PseudoLabelDataset(Dataset):
    def __init__(self, img_dir, pseudo_mask_dir, transform=None):
        self.img_dir        = img_dir
        self.pseudo_mask_dir = pseudo_mask_dir
        self.transform      = transform
        self.ids            = sorted([os.path.splitext(f)[0]
                                      for f in os.listdir(pseudo_mask_dir) if f.endswith('.png')])

        self.rare_classes_in_image = {img_id: set() for img_id in self.ids}
        self.water_in_image        = set()
        for img_id in self.ids:
            mask   = np.array(Image.open(os.path.join(pseudo_mask_dir, f"{img_id}.png")))
            unique = set(np.unique(mask)) - {255}
            self.rare_classes_in_image[img_id] = unique & RARE_CLASSES
            if 9 in unique:
                self.water_in_image.add(img_id)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img    = np.array(Image.open(os.path.join(self.img_dir, f"{img_id}.jpg")).convert('RGB'))
        mask   = np.array(Image.open(os.path.join(self.pseudo_mask_dir, f"{img_id}.png")))
        if self.transform:
            transformed = self.transform(image=img, mask=mask)
            img  = transformed['image']
            mask = transformed['mask'].long()
        return img, mask, img_id

    def get_rare_indices(self):
        return [i for i, img_id in enumerate(self.ids)
                if len(self.rare_classes_in_image.get(img_id, set())) > 0]

    def get_water_indices(self):
        return [i for i, img_id in enumerate(self.ids) if img_id in self.water_in_image]

    def get_normal_indices(self):
        rare = set(self.get_rare_indices())
        return [i for i in range(len(self.ids)) if i not in rare]


print("Dataset classes ready")

Dataset classes ready


In [37]:
import random
from torch.utils.data import BatchSampler

# ==================================================
# CELL 7: Balanced Batch Sampler
# ==================================================

class BalancedBatchSampler(BatchSampler):
    def __init__(self, dataset, batch_size, rare_ratio=0.40, water_ratio=0.15, drop_last=True):
        self.dataset       = dataset
        self.batch_size    = batch_size
        self.rare_ratio    = rare_ratio
        self.water_ratio   = water_ratio
        self.drop_last     = drop_last
        self.rare_indices  = dataset.get_rare_indices()
        self.water_indices = dataset.get_water_indices()

        # [FIX] Mencegah kemungkinan error pop() jika tidak ada sample kelas air sama sekali
        if len(self.water_indices) == 0:
            print("[WARN] Kelas water tidak ditemukan, water_ratio diset ke 0.")
            self.water_ratio = 0.0

        self.water_set     = set(self.water_indices)
        self.normal_indices = dataset.get_normal_indices()

        # FIX #5: n_batches dihitung dari total dataset size, bukan bergantung
        # pada pool yang di-refill. Ini membuat __len__ akurat dan
        # scheduler step count benar.
        self._n_batches = len(self.dataset) // self.batch_size
        print(f"Sampler: {len(self.rare_indices)} rare, "
              f"{len(self.water_indices)} water, "
              f"{len(self.normal_indices)} normal | "
              f"{self._n_batches} batches/epoch")

    def __iter__(self):
        water_pool  = self.water_indices.copy()
        rare_pool   = [i for i in self.rare_indices if i not in self.water_set]
        normal_pool = self.normal_indices.copy()

        random.shuffle(water_pool)
        random.shuffle(rare_pool)
        random.shuffle(normal_pool)

        for _ in range(self._n_batches):
            batch = []
            for _ in range(self.batch_size):
                r = random.random()
                if r < self.water_ratio:
                    if not water_pool:
                        water_pool = self.water_indices.copy()
                        random.shuffle(water_pool)
                    batch.append(water_pool.pop())
                # [FIX] Akumulasi probabilitas agar porsi rare menjadi benar-benar 40% (atau sesuai rare_ratio)
                elif r < self.water_ratio + self.rare_ratio:
                    if not rare_pool:
                        rare_pool = [i for i in self.rare_indices if i not in self.water_set]
                        random.shuffle(rare_pool)
                    batch.append(rare_pool.pop())
                else:
                    if not normal_pool:
                        normal_pool = self.normal_indices.copy()
                        random.shuffle(normal_pool)
                    batch.append(normal_pool.pop())
            yield batch

    def __len__(self):
        return self._n_batches  # FIX #5: konsisten dengan _n_batches


print("BalancedBatchSampler ready")

BalancedBatchSampler ready


In [38]:
import torch
from torch.utils.data import DataLoader

# ==================================================
# CELL 8: Create DataLoaders
# ==================================================

# FIX #3: Definisikan train_ids & val_ids di sini secara eksplisit
# agar semua cell downstream tidak bergantung pada state dari cell lain
train_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
val_ids   = sorted([os.path.splitext(f)[0] for f in os.listdir(VAL_IMG)   if f.endswith('.jpg')])

train_dataset = FloodSegDataset(TRAIN_IMG, TRAIN_MASK, transform=train_transform, image_ids=train_ids)
val_dataset   = FloodSegDataset(VAL_IMG,   VAL_MASK,   transform=val_transform,   image_ids=val_ids)
test_dataset  = FloodTestDataset(TEST_IMG)

num_workers = 2

train_sampler = BalancedBatchSampler(
    train_dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    rare_ratio=TRAIN_CONFIG['rare_ratio'],
    water_ratio=TRAIN_CONFIG['water_ratio'],
    drop_last=True
)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(
    train_dataset, batch_sampler=train_sampler,
    num_workers=num_workers, pin_memory=True,
    worker_init_fn=seed_worker, generator=g
)
val_loader = DataLoader(
    val_dataset, batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=False, num_workers=num_workers, pin_memory=True,
    worker_init_fn=seed_worker, generator=g
)

print(f"Train : {len(train_dataset)} images → {len(train_loader)} batches")
print(f"Val   : {len(val_dataset)} images → {len(val_loader)} batches")
print(f"Test  : {len(test_dataset)} images")

Indexing:   0%|          | 0/1444 [00:00<?, ?it/s]

Indexing:   0%|          | 0/448 [00:00<?, ?it/s]

Sampler: 927 rare, 116 water, 517 normal | 722 batches/epoch
Train : 1444 images → 722 batches
Val   : 448 images → 224 batches
Test  : 447 images


In [39]:
# ==================================================
# CELL 9: Model Definition
# ==================================================
from transformers import SegformerForSemanticSegmentation

class Mask2FormerFlood(nn.Module):
    def __init__(self, num_classes=10, model_name=MODEL_NAME):
        super().__init__()
        self.model = Mask2FormerForUniversalSegmentation.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True,
            id2label={i: CLASS_NAMES[i] for i in range(num_classes)},
            label2id={CLASS_NAMES[i]: i for i in range(num_classes)},
        )

    def forward(self, pixel_values):
        outputs     = self.model(pixel_values=pixel_values)
        masks       = outputs.masks_queries_logits   # (B, N, H/4, W/4)
        class_logits = outputs.class_queries_logits  # (B, N, C+1)

        temperature  = 0.5
        class_probs  = F.softmax(class_logits / temperature, dim=-1)[:, :, :-1]  # drop no-obj
        mask_probs   = masks.sigmoid()

        B, N, C     = class_probs.shape
        _, _, H, W  = mask_probs.shape

        class_probs_rs  = class_probs.permute(0, 2, 1)          # (B, C, N)
        mask_probs_flat = mask_probs.view(B, N, H * W)           # (B, N, H*W)
        seg_maps        = torch.bmm(class_probs_rs, mask_probs_flat).view(B, C, H, W)  # (B, C, H, W)

        seg_maps = F.interpolate(
            seg_maps,
            size=pixel_values.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        # Normalisasi seg_maps agar total probabilitas per pixel = 1
        seg_maps = seg_maps / seg_maps.sum(dim=1, keepdim=True).clamp(min=1e-8)
        seg_maps = seg_maps.clamp(min=1e-8)
        return torch.log(seg_maps)  # (B, C, H, W) — Normalized log-probability map


def init_model(device='cuda'):
    model = Mask2FormerFlood(num_classes=NUM_CLASSES)
    return model.to(device)


def load_m2f(ckpt_path, device='cuda'):
    model = Mask2FormerFlood(num_classes=NUM_CLASSES).to(device)

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

        # [CRITICAL FIX] Prioritaskan load bobot EMA agar mendapat model yang lebih robust (smooth)
        if 'ema_state' in ckpt:
            model.load_state_dict(ckpt['ema_state'])
            status = "EMA Weights"
        else:
            model.load_state_dict(ckpt['model_state_dict'])
            status = "Normal Weights"

        best = ckpt.get('best_score', ckpt.get('best_miou', 0))
        print(f"[OK] M2F loaded ({status}) | epoch {ckpt.get('epoch','?')} | best {best:.4f}")
        return model, ckpt
    else:
        print(f"[WARN] Checkpoint not found: {ckpt_path}")
        return model, None


class SegFormerFlood(nn.Module):
    """SegFormer-B4 wrapper — output raw logits (NOT log-prob)."""
    def __init__(self, num_classes=10, model_name=SEGFORMER_NAME):
        super().__init__()
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True,
            id2label={i: CLASS_NAMES[i] for i in range(num_classes)},
            label2id={CLASS_NAMES[i]: i for i in range(num_classes)},
        )

    def forward(self, pixel_values):
        outputs = self.model(pixel_values=pixel_values)
        logits  = outputs.logits  # (B, C, H/4, W/4) — raw logits
        logits  = F.interpolate(
            logits,
            size=pixel_values.shape[-2:],
            mode='bilinear',
            align_corners=False
        )
        return logits  # raw logits, consumer applies softmax


def load_segformer(ckpt_path, device='cuda'):
    model = SegFormerFlood(num_classes=NUM_CLASSES).to(device)
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        best = ckpt.get('best_score', ckpt.get('best_miou', 0))
        print(f"[OK] SegFormer loaded | epoch {ckpt.get('epoch','?')} | best {best:.4f}")
        return model, ckpt
    else:
        print(f"[WARN] SegFormer checkpoint not found: {ckpt_path}")
        return model, None


print("Model classes ready")
print("[INFO] EMA Auto-Loading is ENABLED!")

Model classes ready
[INFO] EMA Auto-Loading is ENABLED!


In [40]:
# ==================================================
# CELL 10: RLE Utilities + Per-Class Thresholding
# ==================================================

def rle_encode(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return ' '.join(str(x) for x in runs)


def mask_to_rle(mask, empty_classes=None):
    if empty_classes is None:
        empty_classes = EMPTY_CLASSES
    result = {}
    for cls_id in range(NUM_CLASSES):
        if cls_id in empty_classes:
            result[cls_id] = ''
        else:
            binary = (mask == cls_id).astype(np.uint8)
            result[cls_id] = rle_encode(binary) if binary.sum() > 0 else ''
    return result


DEFAULT_CLASS_THRESHOLDS = {
    0: 0.5,  # background
    1: 0.5,  # building flooded
    3: 0.5,  # grass
    4: 0.4,  # pool
    5: 0.5,  # road flooded
    7: 0.5,  # tree
    8: 0.3,  # vehicle
    9: 0.3,  # water
}


def predict_with_per_class_threshold(probs, class_thresholds=None):
    """
    probs : torch.Tensor (C, H, W) — HARUS berupa probability (bukan log-prob).
            Kalau dari M2F: torch.exp(model_output).
            Kalau dari SegFormer: F.softmax(model_output, dim=0).
    Returns: numpy array (H, W) dengan class ID.
    """
    if class_thresholds is None:
        class_thresholds = DEFAULT_CLASS_THRESHOLDS

    C, H, W = probs.shape
    device  = probs.device

    thresh_tensor = torch.ones((C, H, W), device=device) * 0.5
    for c, t in class_thresholds.items():
        if c < C:
            thresh_tensor[c] = t

    margin       = probs - thresh_tensor
    mask         = torch.argmax(margin, dim=0)

    # Pixel yang tidak ada classnya melebihi threshold → fallback background
    max_margin, _ = torch.max(margin, dim=0)
    mask[max_margin < 0] = 0

    # Paksa empty classes ke background
    for cls_id in EMPTY_CLASSES:
        mask[mask == cls_id] = 0

    return mask.cpu().numpy()


print("RLE + per-class thresholding ready")


RLE + per-class thresholding ready


In [41]:
# ==================================================
# CELL 11: Metrics & Optimizer
# ==================================================

class FloodSegMetrics:
    def __init__(self, num_classes, ignore_classes=None):
        self.num_classes    = num_classes
        self.ignore_classes = ignore_classes if ignore_classes else set()
        self.reset()

    def reset(self):
        self.confusion_matrix = np.zeros(
            (self.num_classes, self.num_classes), dtype=np.int64
        )

    def update(self, preds, targets):
        preds   = preds.flatten()
        targets = targets.flatten()
        # Exclude ignore_index=255 dan empty classes dari metric
        valid   = (targets != 255) & (~np.isin(targets, list(self.ignore_classes)))
        preds, targets = preds[valid], targets[valid]
        indices = self.num_classes * targets + preds
        m = np.bincount(indices, minlength=self.num_classes ** 2)
        self.confusion_matrix += m.reshape(self.num_classes, self.num_classes)

    def compute_iou(self):
        intersection = np.diag(self.confusion_matrix)
        union        = (self.confusion_matrix.sum(axis=1)
                        + self.confusion_matrix.sum(axis=0)
                        - intersection)
        # [FIX] Gunakan np.nan untuk kelas absen agar tidak menghukum skor mIoU
        iou          = np.full(self.num_classes, np.nan)
        valid        = union > 0
        iou[valid]   = intersection[valid] / union[valid]
        return iou

    def compute_miou(self):
        ious         = self.compute_iou()
        valid_classes = ~np.isin(np.arange(self.num_classes), list(self.ignore_classes))
        # [FIX] Gunakan np.nanmean
        return np.nanmean(ious[valid_classes])

    def compute_dice(self):
        intersection = np.diag(self.confusion_matrix)
        sum_rows     = self.confusion_matrix.sum(axis=1)
        sum_cols     = self.confusion_matrix.sum(axis=0)
        # [FIX] Gunakan np.nan untuk konsistensi
        dice         = np.full(self.num_classes, np.nan)
        valid        = (sum_rows + sum_cols) > 0
        dice[valid]  = 2 * intersection[valid] / (sum_rows[valid] + sum_cols[valid])
        return dice


def configure_optimizer(model, lr_backbone=3e-5, lr_head=1e-4, weight_decay=0.01):
    backbone_params, head_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if 'pixel_level_module.encoder' in n:
            backbone_params.append(p)
        else:
            head_params.append(p)
    param_groups = [
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': head_params,     'lr': lr_head},
    ]
    print(f"Optimizer: {len(backbone_params)} backbone (lr={lr_backbone}), "
          f"{len(head_params)} head (lr={lr_head})")
    return AdamW(param_groups, weight_decay=weight_decay)


def get_poly_scheduler(optimizer, num_epochs, steps_per_epoch,
                        accum_steps=1, power=0.9, warmup_steps=500):
    total_optimizer_steps = num_epochs * (steps_per_epoch // accum_steps)

    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return max(1e-6, current_step / max(warmup_steps, 1))
        adjusted         = current_step - warmup_steps
        total_after_warm = max(total_optimizer_steps - warmup_steps, 1)
        return max(0.0, (1 - adjusted / total_after_warm) ** power)

    print(f"Scheduler: {total_optimizer_steps} total optimizer steps, "
          f"{warmup_steps} warmup")
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


print("Metrics & optimizer ready")


Metrics & optimizer ready


In [42]:
# ==================================================
# CELL 12: Trainer Class (REVISED: OOM Prevention)
# ==================================================

class EMAModel:
    """Exponential Moving Average of model weights."""
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = {k: v.clone().float() for k, v in model.state_dict().items()}
        self.model  = model
        self._backup = {}

    def update(self):
        with torch.no_grad():
            for k, v in self.model.state_dict().items():
                self.shadow[k] = self.decay * self.shadow[k] + (1.0 - self.decay) * v.float()

    def apply_shadow(self):
        self._backup = {k: v.clone() for k, v in self.model.state_dict().items()}
        state = {k: v.to(next(self.model.parameters()).device)
                 for k, v in self.shadow.items()}
        self.model.load_state_dict(state)

    def restore(self):
        if self._backup:
            self.model.load_state_dict(self._backup)

    def state_dict(self):
        return self.shadow

    def load_state_dict(self, state):
        self.shadow = {k: v.clone().float() for k, v in state.items()}


class Trainer:
    def __init__(self, model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, device, config,
                 use_amp=True, task_name='baseline'):
        self.model        = model
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.loss_fn      = loss_fn
        self.optimizer    = optimizer
        self.scheduler    = scheduler
        self.device       = device
        self.config       = config
        self.use_amp      = use_amp and device.type == 'cuda'
        self.scaler       = torch.cuda.amp.GradScaler(enabled=True) if self.use_amp else None
        self.metrics      = FloodSegMetrics(NUM_CLASSES, EMPTY_CLASSES)
        self.best_score   = 0.0
        self.ema          = EMAModel(model, decay=0.999)
        self.best_smoothed_val_loss = float('inf')
        self.patience_counter       = 0
        self.history      = {
            'train_loss': [], 'val_loss': [], 'val_miou': [],
            'lr_backbone': [], 'lr_head': [], 'class_iou': []
        }
        self.batch_size    = config['batch_size']
        self.accum_steps   = config.get('accumulation_steps', 1)
        self.steps_per_epoch = len(train_loader)

        save_dir = config['save_dir']
        self.last_checkpoint_path = os.path.join(save_dir, f'last_checkpoint_{task_name}.pth')
        self.best_checkpoint_path = os.path.join(save_dir, f'best_miou_{task_name}.pth')
        self.best_loss_path       = os.path.join(save_dir, f'best_loss_{task_name}.pth')

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        pbar       = tqdm(self.train_loader, desc=f'Epoch {epoch}')
        self.optimizer.zero_grad()

        for batch_idx, (images, masks, _) in enumerate(pbar):
            images = images.to(self.device)
            masks  = masks.to(self.device)

            if self.use_amp:
                with torch.cuda.amp.autocast():
                    log_probs = self.model(images)
                    loss, ce_loss, lovasz_loss = self.loss_fn(log_probs, masks)
                loss = loss / self.accum_steps
                self.scaler.scale(loss).backward()

                if (batch_idx + 1) % self.accum_steps == 0:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), self.config['grad_clip']
                    )
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.scheduler.step()
                    self.optimizer.zero_grad()
                    self.ema.update()
            else:
                log_probs = self.model(images)
                loss, ce_loss, lovasz_loss = self.loss_fn(log_probs, masks)
                loss = loss / self.accum_steps
                loss.backward()

                if (batch_idx + 1) % self.accum_steps == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), self.config['grad_clip']
                    )
                    self.optimizer.step()
                    self.scheduler.step()
                    self.optimizer.zero_grad()
                    self.ema.update()

            total_loss   += loss.item() * self.accum_steps
            lr_backbone   = self.optimizer.param_groups[0]['lr']
            lr_head       = self.optimizer.param_groups[1]['lr']
            pbar.set_postfix({
                'loss':  f'{loss.item()*self.accum_steps:.4f}',
                'lr_b':  f'{lr_backbone:.2e}',
                'lr_h':  f'{lr_head:.2e}',
            })

        if (batch_idx + 1) % self.accum_steps != 0:
            if self.use_amp:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.config['grad_clip']
                )
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.config['grad_clip']
                )
                self.optimizer.step()
            self.scheduler.step()
            self.optimizer.zero_grad()
            self.ema.update()

        return total_loss / len(self.train_loader)

    @torch.no_grad()
    def validate(self, verbose=True):
        self.ema.apply_shadow()
        self.model.eval()

        # [FIX OOM] Bersihkan sampah memori dari RAM & VRAM
        torch.cuda.empty_cache()
        import gc
        gc.collect()

        self.metrics.reset()
        total_loss = 0
        pbar = tqdm(self.val_loader, desc='Validating', leave=False)

        for images, masks, _ in pbar:
            images = images.to(self.device)
            masks  = masks.to(self.device)

            with torch.cuda.amp.autocast(enabled=self.use_amp):
                log_probs = self.model(images)
                loss, _, _ = self.loss_fn(log_probs, masks)
            total_loss += loss.item()

            preds = torch.argmax(log_probs, dim=1)
            self.metrics.update(preds.cpu().numpy(), masks.cpu().numpy())

        avg_loss = total_loss / len(self.val_loader)
        miou     = self.metrics.compute_miou()
        ious     = self.metrics.compute_iou()
        dice     = self.metrics.compute_dice()

        if verbose:
            print("\n" + "="*65)
            print(f"{'Class':<20} {'IoU':>8} {'Dice':>8}")
            print("-"*40)
            for c in range(NUM_CLASSES):
                if c in EMPTY_CLASSES:
                    continue
                i_str = f"{ious[c]:>8.3f}" if not np.isnan(ious[c]) else "     NaN"
                d_str = f"{dice[c]:>8.3f}" if not np.isnan(dice[c]) else "     NaN"
                print(f"{CLASS_NAMES[c][:18]:<20} {i_str} {d_str}")
            print("-"*40)
            print(f"{'mIoU':<20} {miou:>8.3f}")
            print("="*65)

        valid_classes = ~np.isin(np.arange(NUM_CLASSES), list(EMPTY_CLASSES))
        mean_dice     = np.nanmean(dice[valid_classes]) if valid_classes.any() else 0.0
        self.ema.restore()
        return avg_loss, miou, mean_dice, ious, dice

    def train(self, start_epoch=1):
        print(f"Training epochs {start_epoch}-{self.config['num_epochs']} | "
              f"Device: {self.device} | AMP: {self.use_amp}")
        print(f"Batch: {self.batch_size} x {self.accum_steps} = "
              f"effective {self.batch_size * self.accum_steps}")

        for epoch in range(start_epoch, self.config['num_epochs'] + 1):
            train_loss                    = self.train_epoch(epoch)
            val_loss, val_miou, val_dice, ious, dice = self.validate(verbose=True)
            lr_backbone = self.optimizer.param_groups[0]['lr']
            lr_head     = self.optimizer.param_groups[1]['lr']

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_miou'].append(val_miou)
            self.history['lr_backbone'].append(lr_backbone)
            self.history['lr_head'].append(lr_head)
            self.history['class_iou'].append(
                {int(k): float(v) if not np.isnan(v) else 0.0 for k, v in enumerate(ious)}
            )

            smoothed_val_loss = (
                np.mean(self.history['val_loss'][-3:])
                if len(self.history['val_loss']) >= 3
                else val_loss
            )

            print(f"Epoch {epoch} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
                  f"mIoU: {val_miou:.4f} | Dice: {val_dice:.4f} | LR: {lr_backbone:.2e}")

            checkpoint = {
                'epoch':                    epoch,
                'model_state_dict':         self.model.state_dict(),
                'optimizer_state_dict':     self.optimizer.state_dict(),
                'scheduler_state_dict':     self.scheduler.state_dict(),
                'best_score':               self.best_score,
                'ema_state':                self.ema.state_dict(),
                'best_smoothed_val_loss':   self.best_smoothed_val_loss,
                'patience_counter':         self.patience_counter,
                'history':                  self.history,
            }
            torch.save(checkpoint, self.last_checkpoint_path)

            val_score = (val_miou + val_dice) / 2.0

            if WANDB_AVAILABLE and wandb.run:
                wandb.log({
                    'train_loss': train_loss, 'val_loss': val_loss,
                    'val_miou': val_miou,     'val_dice': val_dice,
                    'val_score': val_score,   'lr': lr_backbone,
                }, step=epoch)

            if val_score > self.best_score:
                self.best_score = val_score
                torch.save(checkpoint, self.best_checkpoint_path)
                print(f"  ✓ New best Score (mIoU+Dice)/2: {val_score:.4f}")

            if smoothed_val_loss < self.best_smoothed_val_loss:
                self.best_smoothed_val_loss = smoothed_val_loss
                self.patience_counter       = 0
                torch.save(checkpoint, self.best_loss_path)
                print(f"  ✓ New best smoothed val_loss: {smoothed_val_loss:.4f}")
            else:
                self.patience_counter += 1
                print(f"  Patience: {self.patience_counter}/{self.config['early_stop_patience']}")

            if self.patience_counter >= self.config['early_stop_patience']:
                print(f"Early stopping at epoch {epoch}")
                break

        history_path = os.path.join(self.config['save_dir'], 'training_history.json')
        with open(history_path, 'w') as f:
            json.dump(self.history, f, indent=2)
        print(f"\nBest Score: {self.best_score:.4f} | History saved → {history_path}")
        return self.history

print("Trainer ready (Anti-OOM with gc.collect() & gradient_checkpointing)")


Trainer ready (Anti-OOM with gc.collect() & gradient_checkpointing)


In [43]:
# ==================================================
# CELL 13: TTA & Inference (Dengan Rotation & Dim Fix)
# ==================================================
import cv2

def apply_rotation(img_np, angle):
    if angle == 0:   return img_np
    if angle == 90:  return np.rot90(img_np, k=-1)
    if angle == 180: return np.rot90(img_np, k=2)
    if angle == 270: return np.rot90(img_np, k=1)
    h, w    = img_np.shape[:2]
    center  = (w // 2, h // 2)
    M       = cv2.getRotationMatrix2D(center, -angle, 1.0)
    return cv2.warpAffine(img_np, M, (w, h), borderMode=cv2.BORDER_REFLECT)


def reverse_rotation_probs(probs_tensor, angle):
    """Reverse rotation on probability map (C, H, W)."""
    if angle == 0:   return probs_tensor
    if angle == 90:  return torch.rot90(probs_tensor, k=1,  dims=[1, 2])
    if angle == 180: return torch.rot90(probs_tensor, k=2,  dims=[1, 2])
    if angle == 270: return torch.rot90(probs_tensor, k=-1, dims=[1, 2])

    C, H, W = probs_tensor.shape
    arr     = probs_tensor.permute(1, 2, 0).cpu().numpy()
    center  = (W // 2, H // 2)
    M       = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = np.stack([
        cv2.warpAffine(arr[:, :, c], M, (W, H), borderMode=cv2.BORDER_REFLECT)
        for c in range(C)
    ], axis=0)
    return torch.from_numpy(rotated).to(probs_tensor.device)


def _model_to_probs(model, tensor, model_type='m2f'):
    output = model(tensor)
    output = F.interpolate(output, size=tensor.shape[-2:],
                           mode='bilinear', align_corners=False)
    if model_type == 'm2f':
        probs = torch.exp(output.float())
    else:
        probs = F.softmax(output.float(), dim=1)
    return probs


@torch.no_grad()
def predict_with_tta(model, img_np, device,
                     scales=[0.75, 1.0, 1.25], rotations=[0],
                     base_size=512, use_flip=True,
                     class_thresholds=None, model_type='m2f'):
    model.eval()
    H_orig, W_orig = img_np.shape[:2]
    all_probs      = []

    for angle in rotations:
        img_rot = apply_rotation(img_np, angle)
        # [FIX] Ambil dimensi SETELAH dirotasi untuk interpolasi
        H_rot, W_rot = img_rot.shape[:2]

        for scale in scales:
            size      = max(32, int(base_size * scale) // 32 * 32)
            transform = A.Compose([
                A.Resize(size, size),
                A.Normalize(mean=MEAN, std=STD),
                ToTensorV2(),
            ])

            for do_flip in ([False, True] if use_flip else [False]):
                img_aug = np.fliplr(img_rot).copy() if do_flip else img_rot
                tensor  = transform(image=img_aug)['image'].unsqueeze(0).to(device)

                probs = _model_to_probs(model, tensor, model_type)
                # [FIX] Interpolasi menggunakan dimensi pasca rotasi (H_rot, W_rot)
                probs = F.interpolate(probs, size=(H_rot, W_rot),
                                      mode='bilinear', align_corners=False)
                probs = probs.squeeze(0)   # (C, H_rot, W_rot)

                if do_flip:
                    probs = torch.flip(probs, dims=[-1])

                # Saat di reverse, ukurannya otomatis kembali ke (H_orig, W_orig)
                probs = reverse_rotation_probs(probs, angle)
                all_probs.append(probs)

    avg_probs = torch.stack(all_probs).mean(dim=0)   # (C, H_orig, W_orig) probability

    if class_thresholds is not None:
        return predict_with_per_class_threshold(avg_probs, class_thresholds)
    else:
        return torch.argmax(avg_probs, dim=0).cpu().numpy()


def generate_submission(model, test_img_dir, device,
                        output_csv='submission.csv',
                        use_tta=True, scales=[0.75, 1.0, 1.25],
                        rotations=[0], class_thresholds=None,
                        model_type='m2f'):
    test_ids = sorted([
        os.path.splitext(f)[0]
        for f in os.listdir(test_img_dir) if f.endswith('.jpg')
    ])
    results  = []
    print(f"Inference: {len(test_ids)} images | TTA: {use_tta} | "
          f"scales: {scales} | rotations: {rotations}")

    for img_id in tqdm(test_ids, desc="Inference"):
        img_np = np.array(
            Image.open(os.path.join(test_img_dir, f"{img_id}.jpg")).convert('RGB')
        )

        if use_tta:
            mask = predict_with_tta(
                model, img_np, device,
                scales=scales, rotations=rotations,
                use_flip=True, class_thresholds=class_thresholds,
                model_type=model_type
            )
        else:
            transform = A.Compose([
                A.Resize(512, 512),
                A.Normalize(mean=MEAN, std=STD),
                ToTensorV2(),
            ])
            tensor = transform(image=img_np)['image'].unsqueeze(0).to(device)
            probs  = _model_to_probs(model, tensor, model_type)
            h, w   = img_np.shape[:2]
            probs  = F.interpolate(probs, size=(h, w), mode='bilinear', align_corners=False)
            mask   = torch.argmax(probs, dim=1).squeeze(0).cpu().numpy()

        if mask.shape != (480, 640):
            mask = np.array(
                Image.fromarray(mask.astype(np.uint8)).resize((640, 480), Image.NEAREST)
            )

        rles = mask_to_rle(mask)
        for class_id in range(NUM_CLASSES):
            results.append({
                'id':             f"{img_id}_{class_id}",
                'encoded_pixels': rles[class_id]
            })

    df = pd.DataFrame(results)
    df['encoded_pixels'] = df['encoded_pixels'].fillna('')
    df.to_csv(output_csv, index=False, na_rep='')
    print(f"Saved: {output_csv} ({len(df)} rows)")
    return df


print("TTA + Inference ready (with correct H_rot/W_rot handling)")


TTA + Inference ready (with correct H_rot/W_rot handling)


In [44]:
# ==================================================
# CELL 14: Validate Submission
# ==================================================

def validate_submission(submission_path='submission.csv'):
    df = pd.read_csv(submission_path)
    df['encoded_pixels'] = df['encoded_pixels'].fillna('')

    assert list(df.columns) == ['id', 'encoded_pixels'], \
        f"Wrong columns: {list(df.columns)}"

    n_images      = len(df['id'].apply(lambda x: x.rsplit('_', 1)[0]).unique())
    expected_rows = n_images * NUM_CLASSES
    assert len(df) == expected_rows, f"Expected {expected_rows}, got {len(df)}"
    assert n_images == 447, f"Expected 447 images, got {n_images}"

    invalid_rle = 0
    for _, row in df.iterrows():
        val = row['encoded_pixels']
        if val and val != '':
            parts = str(val).split()
            if len(parts) % 2 != 0:
                invalid_rle += 1

    print(f"Rows: {len(df)}, Images: {n_images}")
    print(f"Invalid RLE: {invalid_rle}")

    for class_id in EMPTY_CLASSES:
        class_rows = df[df['id'].str.endswith(f'_{class_id}')]
        non_empty  = len(class_rows[class_rows['encoded_pixels'] != ''])
        status     = '[WARN]' if non_empty > 0 else '[OK]'
        print(f"Class {class_id} ({CLASS_NAMES[class_id]}): "
              f"{non_empty} non-empty {status}")

    if invalid_rle == 0:
        print("[OK] SUBMISSION VALID")
        return True
    else:
        print("[FAIL] SUBMISSION INVALID")
        return False


print("Validation function ready")


Validation function ready


In [50]:
# ==================================================
# CELL 15: BASELINE TRAINING (Real Lovasz-Softmax)
# ==================================================
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import shutil

# ==================================================
# TRUE LOVASZ-SOFTMAX IMPLEMENTATION
# ==================================================
def lovasz_grad(gt_sorted):
    """Menghitung gradien Lovasz extension"""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def lovasz_softmax_loss(probas, labels, classes, ignore=255):
    """
    probas: Tensor (B, C, H, W) berisi probabilitas (setelah softmax)
    labels: Tensor (B, H, W) berisi target class
    classes: list integer ID kelas yang valid untuk dihitung
    """
    B, C, H, W = probas.size()
    probas = probas.permute(0, 2, 3, 1).contiguous().view(-1, C)  # (B*H*W, C)
    labels = labels.view(-1)

    if ignore is not None:
        valid = (labels != ignore)
        if valid.sum() == 0:
            return probas.sum() * 0.0
        probas = probas[valid]
        labels = labels[valid]

    losses = []
    for c in classes:
        fg = (labels == c).float()
        if fg.sum() == 0: # Abaikan jika target kelas ini tidak ada di gambar
            continue
        class_pred = probas[:, c]
        errors = (fg - class_pred).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg_sorted)))

    if len(losses) == 0:
        return probas.sum() * 0.0
    return sum(losses) / len(losses)


class FloodSegLoss(nn.Module):
    def __init__(self, class_weights, lovasz_weight=0.75, empty_classes=None):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)
        self.lovasz_weight = lovasz_weight
        self.empty_classes = empty_classes if empty_classes is not None else set()

    def forward(self, log_probs, masks):
        # 1. Cross-Entropy Loss
        ce_loss_val = self.ce_loss(log_probs, masks)

        # 2. Lovasz-Softmax Loss
        lovasz_loss_val = torch.tensor(0.0, device=log_probs.device, dtype=log_probs.dtype)
        if self.lovasz_weight > 0:
            probas = F.softmax(log_probs, dim=1)
            # Filter agar kelas yang "kosong" (seperti indeks 2 & 6) tidak diikutkan
            valid_classes = [c for c in range(probas.shape[1]) if c not in self.empty_classes]
            lovasz_loss_val = lovasz_softmax_loss(probas, masks, classes=valid_classes, ignore=255)

        total_loss = (1 - self.lovasz_weight) * ce_loss_val + self.lovasz_weight * lovasz_loss_val
        return total_loss, ce_loss_val, lovasz_loss_val


# ==================================================
# TRAINING SCRIPT
# ==================================================
device = torch.device('cuda')
torch.cuda.empty_cache()

train_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
val_ids   = sorted([os.path.splitext(f)[0] for f in os.listdir(VAL_IMG)   if f.endswith('.jpg')])

# 1. CEK CHECKPOINT
last_checkpoint = os.path.join(TRAIN_CONFIG['save_dir'], 'last_checkpoint_baseline.pth')
checkpoint      = None
start_epoch     = 1

if os.path.exists(last_checkpoint):
    print(f"[OK] Found checkpoint: {last_checkpoint}")
    checkpoint  = torch.load(last_checkpoint, map_location='cpu', weights_only=False)
    start_epoch = checkpoint.get('epoch', 0) + 1
    print(f"     Resume from epoch {start_epoch-1}, "
          f"best score: {checkpoint.get('best_score', 0):.4f}")
else:
    print("[INFO] No checkpoint found, training from scratch")

# 2. MODEL
model = init_model(device)
if checkpoint is not None:
    model.load_state_dict(checkpoint['model_state_dict'])
    print("[OK] Model weights loaded")

# 3. OPTIMIZER
optimizer = configure_optimizer(
    model,
    lr_backbone=TRAIN_CONFIG['lr_backbone'],
    lr_head=TRAIN_CONFIG['lr_head'],
    weight_decay=TRAIN_CONFIG['weight_decay']
)

# 4. DATALOADER
train_ds = FloodSegDataset(TRAIN_IMG, TRAIN_MASK, transform=train_transform, image_ids=train_ids)
val_ds   = FloodSegDataset(VAL_IMG,   VAL_MASK,   transform=val_transform,   image_ids=val_ids)

sampler_bl = BalancedBatchSampler(
    train_ds,
    batch_size=TRAIN_CONFIG['batch_size'],
    rare_ratio=TRAIN_CONFIG['rare_ratio'],
    water_ratio=TRAIN_CONFIG['water_ratio'],
    drop_last=True
)
train_loader_bl = DataLoader(train_ds, batch_sampler=sampler_bl,
                              num_workers=2, pin_memory=True)
val_loader_bl   = DataLoader(val_ds, batch_size=TRAIN_CONFIG['batch_size'],
                              shuffle=False, num_workers=2, pin_memory=True)

# 5. SCHEDULER
scheduler = get_poly_scheduler(
    optimizer,
    TRAIN_CONFIG['num_epochs'],
    len(train_loader_bl),
    accum_steps=TRAIN_CONFIG['accumulation_steps'],
    power=TRAIN_CONFIG['poly_power'],
    warmup_steps=TRAIN_CONFIG['warmup_steps']
)

if checkpoint is not None:
    try:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print("[OK] Optimizer state loaded")
    except Exception as e:
        print(f"[WARN] Could not load optimizer state: {e}")

    steps_per_epoch_actual = math.ceil(len(train_loader_bl) / TRAIN_CONFIG['accumulation_steps'])
    steps_to_catch_up = (start_epoch - 1) * steps_per_epoch_actual
    for _ in range(steps_to_catch_up):
        scheduler.step()
    print(f"[OK] Scheduler fast-forwarded {steps_to_catch_up} steps")

# 6. W&B
if WANDB_AVAILABLE:
    wandb.init(
        project="flood-segmentation",
        name=f"m2f-baseline-{TRAIN_CONFIG['num_epochs']}ep",
        config={**TRAIN_CONFIG, 'model': MODEL_NAME},
        resume="allow",
    )
    print("[OK] W&B initialized")
else:
    print("[WARN] W&B not available")

# 7. LOSS
loss_fn = FloodSegLoss(
    CLASS_WEIGHTS.to(device),
    lovasz_weight=TRAIN_CONFIG['lovasz_weight'],
    empty_classes=EMPTY_CLASSES
)

# 8. TRAINER
trainer = Trainer(
    model=model, train_loader=train_loader_bl, val_loader=val_loader_bl,
    loss_fn=loss_fn, optimizer=optimizer, scheduler=scheduler,
    device=device, config=TRAIN_CONFIG, use_amp=True,
    task_name='baseline'
)

# 9. RESTORE TRAINER STATE
if checkpoint is not None:
    trainer.best_score              = checkpoint.get('best_score', 0.0)
    trainer.best_smoothed_val_loss  = checkpoint.get('best_smoothed_val_loss', float('inf'))
    trainer.patience_counter        = checkpoint.get('patience_counter', 0)
    trainer.history                 = checkpoint.get('history', {
        'train_loss': [], 'val_loss': [], 'val_miou': [],
        'lr_backbone': [], 'lr_head': [], 'class_iou': []
    })
    if 'ema_state' in checkpoint:
        trainer.ema.load_state_dict(checkpoint['ema_state'])
    print(f"[OK] Trainer state restored (best_score: {trainer.best_score:.4f})")

# 10. TRAIN
print(f"\n{'='*60}")
print(f"STARTING TRAINING FROM EPOCH {start_epoch}")
print(f"{'='*60}\n")

history = trainer.train(start_epoch=start_epoch)
print(f"\n[OK] Training done! Best score: {trainer.best_score:.4f}")

if os.path.exists(trainer.best_checkpoint_path):
    shutil.copy(trainer.best_checkpoint_path, CKPT_M2F)
    print(f"[OK] Best checkpoint copied \u2192 {CKPT_M2F}")


[INFO] No checkpoint found, training from scratch


Loading weights:   0%|          | 0/782 [00:00<?, ?it/s]

Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-base-ade-semantic
Key                    | Status   |                                                                                        
-----------------------+----------+----------------------------------------------------------------------------------------
criterion.empty_weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([151]) vs model:torch.Size([11])          
class_predictor.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([151, 256]) vs model:torch.Size([11, 256])
class_predictor.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([151]) vs model:torch.Size([11])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Optimizer: 429 backbone (lr=3e-05), 328 head (lr=0.0001)


Indexing:   0%|          | 0/1444 [00:00<?, ?it/s]

...

Epoch 3:   0%|          | 0/722 [00:00<?, ?it/s]

Validating:   0%|          | 0/224 [00:00<?, ?it/s]


Class                     IoU     Dice
----------------------------------------
background              0.118    0.212
building flooded        0.195    0.326
grass                   0.017    0.033
pool                    0.030    0.058
road flooded            0.114    0.205
tree                    0.111    0.200
vehicle                 0.004    0.008
water                   0.004    0.007
----------------------------------------
mIoU                    0.074
Epoch 3 | Train: 0.6805 | Val: 1.1754 | mIoU: 0.0741 | Dice: 0.1312 | LR: 1.64e-05
  ✓ New best Score (mIoU+Dice)/2: 0.1026
  ✓ New best smoothed val_loss: 1.2498


Epoch 4:   0%|          | 0/722 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ==================================================
# CELL 16: Baseline Inference & Submission
# ==================================================

device = torch.device('cuda')
torch.cuda.empty_cache()

if os.path.exists(CKPT_M2F):
    model, ckpt = load_m2f(CKPT_M2F, device)
    print(f"Loaded epoch {ckpt['epoch']}, "
          f"best score: {ckpt.get('best_score', ckpt.get('best_miou', 0)):.4f}")
else:
    print(f"[WARN] Checkpoint not found: {CKPT_M2F} — using untrained model")
    model = init_model(device)

df = generate_submission(
    model=model,
    test_img_dir=TEST_IMG,
    device=device,
    output_csv='submission_baseline.csv',
    use_tta=True,
    scales=[0.75, 1.0, 1.25],
    rotations=[0],
    class_thresholds=None,
    model_type='m2f'
)

validate_submission('submission_baseline.csv')


In [ ]:
# ==================================================
# CELL 17: Download Submission
# ==================================================

from google.colab import files

for fname in ['submission_baseline.csv']:
    if os.path.exists(fname):
        print(f"Downloading {fname}...")
        files.download(fname)
    else:
        print(f"[WARN] {fname} not found, skip")


In [ ]:
# ==================================================
# CELL 18: TASK 5 - Pseudo-Labeling (OPTIMIZED)
# ==================================================
import cv2

train_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
val_ids   = sorted([os.path.splitext(f)[0] for f in os.listdir(VAL_IMG)   if f.endswith('.jpg')])

CLASS_PSEUDO_THRESHOLDS = {
    0: 0.95,  # background
    1: 0.90,  # building flooded
    3: 0.90,  # grass
    4: 0.85,  # pool
    5: 0.90,  # road flooded
    7: 0.90,  # tree
    8: 0.75,  # vehicle
    9: 0.75,  # water
}


def remove_small_components(mask_np, min_size=100):
    cleaned      = np.copy(mask_np)
    unique_classes = np.unique(mask_np)
    for c in unique_classes:
        if c == 0 or c == 255:
            continue
        binary_mask = (mask_np == c).astype(np.uint8)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            binary_mask, connectivity=8
        )
        for i in range(1, num_labels):
            if stats[i, cv2.CC_STAT_AREA] < min_size:
                cleaned[labels == i] = 0
    return cleaned


def apply_adaptive_threshold(pred_mask, max_probs, class_thresholds):
    pseudo_mask = pred_mask.clone()

    for class_id, threshold in class_thresholds.items():
        if class_id in EMPTY_CLASSES:
            continue
        low_conf = (pred_mask == class_id) & (max_probs < threshold)
        pseudo_mask[low_conf] = 255  # ignore_index di loss

    for cls_id in EMPTY_CLASSES:
        pseudo_mask[pseudo_mask == cls_id] = 255

    return pseudo_mask


@torch.no_grad()
def generate_pseudo_labels_optimized(
    model_m2f, model_seg, test_img_dir, output_dir, device,
    weight_m2f=0.65, scales=[0.75, 1.0, 1.25], rotations=[0]
):
    model_m2f.eval()
    if model_seg is not None:
        model_seg.eval()

    test_ids    = sorted([
        os.path.splitext(f)[0]
        for f in os.listdir(test_img_dir) if f.endswith('.jpg')
    ])
    accepted    = 0
    total_pix   = 0
    conf_pix    = 0
    class_counts = {c: 0 for c in range(NUM_CLASSES)}

    for img_id in tqdm(test_ids, desc="Generating Pseudo-Labels"):
        img_np   = np.array(
            Image.open(os.path.join(test_img_dir, f"{img_id}.jpg")).convert('RGB')
        )
        H, W     = img_np.shape[:2]
        all_probs = []

        for angle in rotations:
            img_rot = apply_rotation(img_np, angle)
            # [FIX] Ekstrak dimensi gambar rotasi
            H_rot, W_rot = img_rot.shape[:2]

            for scale in scales:
                size      = max(32, int(512 * scale) // 32 * 32)
                transform = A.Compose([
                    A.Resize(size, size),
                    A.Normalize(mean=MEAN, std=STD),
                    ToTensorV2(),
                ])

                for do_flip in [False, True]:
                    img_aug = np.fliplr(img_rot).copy() if do_flip else img_rot
                    tensor  = transform(image=img_aug)['image'].unsqueeze(0).to(device)

                    probs_m2f = _model_to_probs(model_m2f, tensor, model_type='m2f')
                    # [FIX] Gunakan H_rot, W_rot untuk interpolasi
                    probs_m2f = F.interpolate(probs_m2f, size=(H_rot, W_rot),
                                              mode='bilinear', align_corners=False)
                    probs_m2f = probs_m2f.squeeze(0)   # (C, H_rot, W_rot)

                    if model_seg is not None:
                        probs_seg = _model_to_probs(model_seg, tensor, model_type='segformer')
                        # [FIX] Gunakan H_rot, W_rot untuk interpolasi
                        probs_seg = F.interpolate(probs_seg, size=(H_rot, W_rot),
                                                  mode='bilinear', align_corners=False)
                        probs_seg = probs_seg.squeeze(0)
                        probs     = weight_m2f * probs_m2f + (1 - weight_m2f) * probs_seg
                    else:
                        probs = probs_m2f

                    if do_flip:
                        probs = torch.flip(probs, dims=[-1])
                    probs = reverse_rotation_probs(probs, angle)
                    all_probs.append(probs)

        avg_probs         = torch.stack(all_probs).mean(dim=0)   # (C, H, W)
        max_probs, pred_mask = torch.max(avg_probs, dim=0)       # (H, W), (H, W)

        pseudo_mask = apply_adaptive_threshold(pred_mask, max_probs, CLASS_PSEUDO_THRESHOLDS)

        total_pix += H * W
        conf_pix  += (pseudo_mask != 255).sum().item()

        for c in range(NUM_CLASSES):
            class_counts[c] += (pseudo_mask == c).sum().item()
        accepted += 1

        pseudo_np = pseudo_mask.cpu().numpy().astype(np.uint8)
        pseudo_np = remove_small_components(pseudo_np, min_size=100)
        Image.fromarray(pseudo_np).save(os.path.join(output_dir, f"{img_id}.png"))

    print(f"Generated {accepted} pseudo-masks")
    print(f"Confident pixels (avg): {conf_pix/max(total_pix,1)*100:.1f}%")
    print("\nClass Distribution in Pseudo-Labels:")
    total_valid = sum(class_counts.values())
    for c in range(NUM_CLASSES):
        if c in EMPTY_CLASSES:
            continue
        pct = (class_counts[c] / max(1, total_valid)) * 100
        print(f"  Class {c} ({CLASS_NAMES[c]}): {pct:.2f}% ({class_counts[c]} px)")
    return accepted


class CombinedDatasetWrapper(Dataset):
    def __init__(self, ds1, ds2):
        self.ds1        = ds1
        self.ds2        = ds2
        self.concat_ds  = torch.utils.data.ConcatDataset([ds1, ds2])
        self.len1       = len(ds1)

    def __len__(self):
        return len(self.concat_ds)

    def __getitem__(self, idx):
        return self.concat_ds[idx]

    def get_rare_indices(self):
        return (self.ds1.get_rare_indices()
                + [i + self.len1 for i in self.ds2.get_rare_indices()])

    def get_water_indices(self):
        return (self.ds1.get_water_indices()
                + [i + self.len1 for i in self.ds2.get_water_indices()])

    def get_normal_indices(self):
        return (self.ds1.get_normal_indices()
                + [i + self.len1 for i in self.ds2.get_normal_indices()])


def retrain_with_pseudo_labels(
    base_ckpt_path, pseudo_mask_dir, save_path,
    config, class_weights, device, iteration=1
):
    last_ckpt   = os.path.join(
        config['save_dir'], f'last_checkpoint_pseudo_iter{iteration}.pth'
    )
    checkpoint  = None
    start_epoch = 1

    if os.path.exists(last_ckpt):
        print(f"[OK] Found pseudo checkpoint iter {iteration}: {last_ckpt}")
        model, checkpoint = load_m2f(last_ckpt, device)
        start_epoch       = checkpoint.get('epoch', 0) + 1
        print(f"     Resume from epoch {start_epoch-1}")
    else:
        model, _ = load_m2f(base_ckpt_path, device)

    _train_ids = sorted([os.path.splitext(f)[0]
                         for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
    _val_ids   = sorted([os.path.splitext(f)[0]
                         for f in os.listdir(VAL_IMG)   if f.endswith('.jpg')])

    orig_dataset   = FloodSegDataset(TRAIN_IMG, TRAIN_MASK,
                                     transform=train_transform, image_ids=_train_ids)
    pseudo_dataset = PseudoLabelDataset(TEST_IMG, pseudo_mask_dir,
                                        transform=train_transform)
    val_ds         = FloodSegDataset(VAL_IMG, VAL_MASK,
                                     transform=val_transform, image_ids=_val_ids)

    combined_dataset = CombinedDatasetWrapper(orig_dataset, pseudo_dataset)
    print(f"Combined iter {iteration}: {len(orig_dataset)} orig + "
          f"{len(pseudo_dataset)} pseudo = {len(combined_dataset)}")

    sampler         = BalancedBatchSampler(
        combined_dataset, batch_size=config['batch_size'],
        rare_ratio=0.40, water_ratio=0.15, drop_last=True
    )
    combined_loader = DataLoader(combined_dataset, batch_sampler=sampler,
                                 num_workers=2, pin_memory=True)
    val_loader_ps   = DataLoader(val_ds, batch_size=config['batch_size'],
                                 shuffle=False, num_workers=2, pin_memory=True)

    optimizer = configure_optimizer(
        model,
        lr_backbone=config['lr_backbone'],
        lr_head=config['lr_head'],
        weight_decay=config['weight_decay']
    )
    scheduler = get_poly_scheduler(
        optimizer, config['num_epochs'], len(combined_loader),
        accum_steps=config['accumulation_steps'],
        power=config['poly_power'],
        warmup_steps=config['warmup_steps']
    )

    if checkpoint is not None:
        try:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            steps_to_catch_up = (
                (start_epoch - 1)
                * (len(combined_loader) // config['accumulation_steps'])
            )
            for _ in range(steps_to_catch_up):
                scheduler.step()
            print(f"[OK] Optimizer loaded, scheduler fast-forwarded "
                  f"{steps_to_catch_up} steps")
        except Exception as e:
            print(f"[WARN] Could not load optimizer state: {e}")

    loss_fn_ps = FloodSegLoss(
        class_weights.to(device),
        lovasz_weight=config['lovasz_weight'],
        empty_classes=EMPTY_CLASSES
    )

    trainer_ps = Trainer(
        model=model, train_loader=combined_loader, val_loader=val_loader_ps,
        loss_fn=loss_fn_ps, optimizer=optimizer, scheduler=scheduler,
        device=device,
        config={**config, 'save_dir': TRAIN_CONFIG['save_dir']},
        use_amp=True, task_name=f'pseudo_iter{iteration}'
    )

    if checkpoint is not None:
        trainer_ps.best_score             = checkpoint.get('best_score', 0.0)
        trainer_ps.best_smoothed_val_loss = checkpoint.get('best_smoothed_val_loss', float('inf'))
        trainer_ps.patience_counter       = checkpoint.get('patience_counter', 0)
        trainer_ps.history                = checkpoint.get('history', {
            'train_loss': [], 'val_loss': [], 'val_miou': [],
            'lr_backbone': [], 'lr_head': [], 'class_iou': []
        })
        print(f"[OK] Trainer state restored (best_score: {trainer_ps.best_score:.4f})")

    print(f"\n{'='*60}\n"
          f"PSEUDO TRAINING ITERATION {iteration} FROM EPOCH {start_epoch}\n"
          f"{'='*60}\n")
    trainer_ps.train(start_epoch=start_epoch)

    torch.save({
        'epoch':                config['num_epochs'], # [FIX] Mengambil epoch limit dari config agar tidak NameError
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_score':           trainer_ps.best_score,
        'history':              trainer_ps.history,
    }, save_path)
    print(f"[OK] Saved pseudo-trained model iter {iteration} → {save_path}")
    return model


# ========== RUN ==========
print("="*60)
print("TASK 5: Optimized Pseudo-Labeling")
print("="*60)

device = torch.device('cuda')
torch.cuda.empty_cache()

teacher_m2f, _  = load_m2f(CKPT_M2F, device)
teacher_seg, _  = (load_segformer(CKPT_SEG, device)
                   if os.path.exists(CKPT_SEG) else (None, None))

NUM_ITERATIONS       = 3
current_teacher_m2f  = teacher_m2f
current_ckpt_path    = CKPT_M2F

final_pseudo_ckpt = CKPT_M2F_PSEUDO

for iteration in range(1, NUM_ITERATIONS + 1):
    print(f"\n--- PSEUDO-LABELING ITERATION {iteration} ---")
    iter_pseudo_dir = f"{PSEUDO_MASK_DIR}_iter{iteration}"
    os.makedirs(iter_pseudo_dir, exist_ok=True)

    generate_pseudo_labels_optimized(
        model_m2f=current_teacher_m2f,
        model_seg=teacher_seg if iteration == 1 else None,
        test_img_dir=TEST_IMG,
        output_dir=iter_pseudo_dir,
        device=device,
        scales=[0.75, 1.0, 1.25],
        rotations=[0]
    )

    iter_ckpt_path = CKPT_M2F_PSEUDO.replace('.pth', f'_iter{iteration}.pth')

    modified_config = TRAIN_CONFIG_FINETUNE.copy()
    modified_config['batch_size']        = 1
    modified_config['accumulation_steps'] = 16

    current_teacher_m2f = retrain_with_pseudo_labels(
        base_ckpt_path=current_ckpt_path,
        pseudo_mask_dir=iter_pseudo_dir,
        save_path=iter_ckpt_path,
        config=modified_config,
        class_weights=CLASS_WEIGHTS_ENHANCED,
        device=device,
        iteration=iteration
    )
    current_ckpt_path = iter_ckpt_path
    final_pseudo_ckpt = iter_ckpt_path

CKPT_M2F_PSEUDO = final_pseudo_ckpt
print(f"\n[OK] Task 5 done! Final pseudo ckpt → {CKPT_M2F_PSEUDO}")


In [ ]:
# ==================================================
# CELL 19: TASK 1 - Ensemble Submission (FULL WORKING)
# ==================================================
print("="*60)
print("TASK 1: Ensemble Submission (SegFormer + Mask2Former)")
print("="*60)

device = torch.device('cuda')
torch.cuda.empty_cache()

model_m2f, ckpt_m2f = load_m2f(CKPT_M2F, device)
model_seg, ckpt_seg = load_segformer(CKPT_SEG, device)

if ckpt_m2f: print(f"M2F best   : {ckpt_m2f.get('best_score', ckpt_m2f.get('best_miou',0)):.4f}")
if ckpt_seg: print(f"SegFormer  : {ckpt_seg.get('best_score', ckpt_seg.get('best_miou',0)):.4f}")

# ========== GRID SEARCH ENSEMBLE WEIGHT ==========
print("\nSearching best ensemble weight on validation set...")
model_m2f.eval()
model_seg.eval()

val_ids        = sorted([os.path.splitext(f)[0]
                          for f in os.listdir(VAL_IMG) if f.endswith('.jpg')])
val_ds_gs      = FloodSegDataset(VAL_IMG, VAL_MASK, transform=val_transform, image_ids=val_ids)
val_loader_gs  = DataLoader(val_ds_gs, batch_size=2, shuffle=False, num_workers=2)

weights_to_test = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
best_weight     = 0.65
best_gs_score   = 0.0

# Cache logits untuk efisiensi
all_logits_m2f, all_logits_seg, all_masks_gs = [], [], []
print("Caching val logits for grid search...")

for images, masks, _ in tqdm(val_loader_gs, leave=False):
    images = images.to(device)
    with torch.no_grad():
        pm = _model_to_probs(model_m2f, images, 'm2f')
        ps = _model_to_probs(model_seg, images, 'segformer')
    all_logits_m2f.append(pm.cpu())
    all_logits_seg.append(ps.cpu())
    all_masks_gs.append(masks.cpu())

print("Evaluating weights...")
for w in weights_to_test:
    metrics = FloodSegMetrics(NUM_CLASSES, EMPTY_CLASSES)
    for pm, ps, masks in zip(all_logits_m2f, all_logits_seg, all_masks_gs):
        probs = w * pm + (1 - w) * ps
        preds = torch.argmax(probs, dim=1)
        metrics.update(preds.numpy(), masks.numpy())
    score = metrics.compute_miou()
    print(f"  Weight M2F: {w:.2f} → mIoU: {score:.4f}")
    if score > best_gs_score:
        best_gs_score = score
        best_weight   = w

print(f"Selected best weight_m2f: {best_weight:.2f} (val mIoU: {best_gs_score:.4f})")

# ========== ENSEMBLE PREDICT ==========
@torch.no_grad()
def ensemble_predict_tta(img_np, weight_m2f=0.65):
    model_m2f.eval()
    model_seg.eval()

    H_orig, W_orig = img_np.shape[:2]
    all_probs      = []
    scales         = [0.75, 0.875, 1.0, 1.125, 1.25]
    rotations      = [0, 90, 180, 270]

    for angle in rotations:
        img_rot = apply_rotation(img_np, angle)
        # [FIX] Ambil dimensi gambar rotasi
        H_rot, W_rot = img_rot.shape[:2]

        for scale in scales:
            size      = max(32, int(512 * scale) // 32 * 32)
            transform = A.Compose([
                A.Resize(size, size),
                A.Normalize(mean=MEAN, std=STD),
                ToTensorV2(),
            ])

            for do_flip in [False, True]:
                img_aug = np.fliplr(img_rot).copy() if do_flip else img_rot
                tensor  = transform(image=img_aug)['image'].unsqueeze(0).to(device)

                pm = _model_to_probs(model_m2f, tensor, 'm2f')
                ps = _model_to_probs(model_seg, tensor, 'segformer')

                # [FIX] Interpolasi menggunakan dimensi pasca-rotasi (H_rot, W_rot)
                pm = F.interpolate(pm, size=(H_rot, W_rot),
                                   mode='bilinear', align_corners=False).squeeze(0)
                ps = F.interpolate(ps, size=(H_rot, W_rot),
                                   mode='bilinear', align_corners=False).squeeze(0)

                probs = weight_m2f * pm + (1.0 - weight_m2f) * ps

                if do_flip:
                    probs = torch.flip(probs, dims=[-1])
                probs = reverse_rotation_probs(probs, angle)
                all_probs.append(probs)

    avg_probs = torch.stack(all_probs).mean(dim=0)
    return predict_with_per_class_threshold(avg_probs, DEFAULT_CLASS_THRESHOLDS)


# ========== GENERATE SUBMISSION ==========
test_ids = sorted([os.path.splitext(f)[0]
                   for f in os.listdir(TEST_IMG) if f.endswith('.jpg')])
results  = []

for img_id in tqdm(test_ids, desc="Ensemble Inference"):
    img_np = np.array(
        Image.open(os.path.join(TEST_IMG, f"{img_id}.jpg")).convert('RGB')
    )
    mask = ensemble_predict_tta(img_np, weight_m2f=best_weight)

    if mask.shape != (480, 640):
        mask = np.array(
            Image.fromarray(mask.astype(np.uint8)).resize((640, 480), Image.NEAREST)
        )

    rles = mask_to_rle(mask)
    for c in range(NUM_CLASSES):
        results.append({'id': f"{img_id}_{c}", 'encoded_pixels': rles[c]})

df = pd.DataFrame(results)
df['encoded_pixels'] = df['encoded_pixels'].fillna('')
df.to_csv('submission_ensemble.csv', index=False, na_rep='')
print(f"[OK] Saved: submission_ensemble.csv ({len(df)} rows)")
validate_submission('submission_ensemble.csv')


In [ ]:
# ==================================================
# CELL 20: TASK 2 - Fine-tune Submission (DENGAN RESUME)
# ==================================================
print("="*60)
print("TASK 2: Fine-tune Mask2Former")
print("="*60)

device = torch.device('cuda')
torch.cuda.empty_cache()

train_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
val_ids   = sorted([os.path.splitext(f)[0] for f in os.listdir(VAL_IMG)   if f.endswith('.jpg')])

# CEK CHECKPOINT
last_checkpoint = os.path.join(TRAIN_CONFIG['save_dir'], 'last_checkpoint_finetune.pth')
checkpoint      = None
start_epoch     = 1

if os.path.exists(last_checkpoint):
    print(f"[OK] Found fine-tune checkpoint: {last_checkpoint}")
    checkpoint  = torch.load(last_checkpoint, map_location='cpu', weights_only=False)
    start_epoch = checkpoint.get('epoch', 0) + 1
    print(f"     Resume from epoch {start_epoch-1}")
else:
    print("[INFO] No checkpoint found, starting from baseline")

# DATALOADER
train_ds = FloodSegDataset(TRAIN_IMG, TRAIN_MASK, transform=train_transform, image_ids=train_ids)
val_ds   = FloodSegDataset(VAL_IMG,   VAL_MASK,   transform=val_transform,   image_ids=val_ids)

sampler_ft      = BalancedBatchSampler(
    train_ds, batch_size=TRAIN_CONFIG_FINETUNE['batch_size'],
    rare_ratio=0.40, water_ratio=0.15, drop_last=True
)
train_loader_ft = DataLoader(train_ds, batch_sampler=sampler_ft,
                              num_workers=2, pin_memory=True)
val_loader_ft   = DataLoader(val_ds, batch_size=2, shuffle=False,
                              num_workers=2, pin_memory=True)

# MODEL
model, _ = load_m2f(CKPT_M2F, device)
if checkpoint is not None:
    model.load_state_dict(checkpoint['model_state_dict'])
    print("[OK] Model weights loaded from checkpoint")

# OPTIMIZER
optimizer = configure_optimizer(
    model,
    lr_backbone=TRAIN_CONFIG_FINETUNE['lr_backbone'],
    lr_head=TRAIN_CONFIG_FINETUNE['lr_head'],
    weight_decay=TRAIN_CONFIG_FINETUNE['weight_decay']
)

# SCHEDULER
scheduler = get_poly_scheduler(
    optimizer,
    TRAIN_CONFIG_FINETUNE['num_epochs'],
    len(train_loader_ft),
    accum_steps=TRAIN_CONFIG_FINETUNE['accumulation_steps'],
    power=TRAIN_CONFIG_FINETUNE['poly_power'],
    warmup_steps=TRAIN_CONFIG_FINETUNE['warmup_steps']
)

if checkpoint is not None:
    try:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print("[OK] Optimizer state loaded")
    except Exception as e:
        print(f"[WARN] Could not load optimizer state: {e}")

    steps_to_catch_up = (
        (start_epoch - 1)
        * (len(train_loader_ft) // TRAIN_CONFIG_FINETUNE['accumulation_steps'])
    )
    for _ in range(steps_to_catch_up):
        scheduler.step()
    print(f"[OK] Scheduler fast-forwarded {steps_to_catch_up} steps")

# LOSS & TRAINER
loss_fn = FloodSegLoss(CLASS_WEIGHTS.to(device), lovasz_weight=0.75,
                        empty_classes=EMPTY_CLASSES)
trainer = Trainer(
    model=model, train_loader=train_loader_ft, val_loader=val_loader_ft,
    loss_fn=loss_fn, optimizer=optimizer, scheduler=scheduler,
    device=device, config=TRAIN_CONFIG_FINETUNE, use_amp=True,
    task_name='finetune'
)

if checkpoint is not None:
    trainer.best_score             = checkpoint.get('best_score', 0.0)
    trainer.best_smoothed_val_loss = checkpoint.get('best_smoothed_val_loss', float('inf'))
    trainer.patience_counter       = checkpoint.get('patience_counter', 0)
    trainer.history                = checkpoint.get('history', {
        'train_loss': [], 'val_loss': [], 'val_miou': [],
        'lr_backbone': [], 'lr_head': [], 'class_iou': []
    })

print(f"\nStarting fine-tune from epoch {start_epoch}")
trainer.train(start_epoch=start_epoch)

torch.save({
    'epoch':                TRAIN_CONFIG_FINETUNE['num_epochs'],
    'model_state_dict':     model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_score':           trainer.best_score,
    'history':              trainer.history,
}, CKPT_M2F_FINETUNE)
print(f"[OK] Saved → {CKPT_M2F_FINETUNE}")

# SUBMISSION
df = generate_submission(
    model=model, test_img_dir=TEST_IMG, device=device,
    output_csv='submission_finetune.csv',
    use_tta=True, scales=[0.75, 1.0, 1.25], rotations=[0],
    class_thresholds=DEFAULT_CLASS_THRESHOLDS,
    model_type='m2f'
)
validate_submission('submission_finetune.csv')
print("[OK] Task 2 completed!")


In [ ]:
# ==================================================
# CELL 21: TASK 3 - Enhanced Class Weights (DENGAN RESUME)
# ==================================================
print("="*60)
print("TASK 3: Enhanced Class Weights (vehicle=25, water=35)")
print("="*60)

device = torch.device('cuda')
torch.cuda.empty_cache()

train_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
val_ids   = sorted([os.path.splitext(f)[0] for f in os.listdir(VAL_IMG)   if f.endswith('.jpg')])

# CEK CHECKPOINT
last_checkpoint = os.path.join(TRAIN_CONFIG['save_dir'], 'last_checkpoint_weighted.pth')
checkpoint      = None
start_epoch     = 1

if os.path.exists(last_checkpoint):
    print(f"[OK] Found weighted checkpoint: {last_checkpoint}")
    checkpoint  = torch.load(last_checkpoint, map_location='cpu', weights_only=False)
    start_epoch = checkpoint.get('epoch', 0) + 1
    print(f"     Resume from epoch {start_epoch-1}")
else:
    print("[INFO] No checkpoint found, starting from baseline")

# DATALOADER
train_ds = FloodSegDataset(TRAIN_IMG, TRAIN_MASK, transform=train_transform, image_ids=train_ids)
val_ds   = FloodSegDataset(VAL_IMG,   VAL_MASK,   transform=val_transform,   image_ids=val_ids)

sampler_wt      = BalancedBatchSampler(
    train_ds, batch_size=2, rare_ratio=0.40, water_ratio=0.15, drop_last=True
)
train_loader_wt = DataLoader(train_ds, batch_sampler=sampler_wt,
                              num_workers=2, pin_memory=True)
val_loader_wt   = DataLoader(val_ds, batch_size=2, shuffle=False,
                              num_workers=2, pin_memory=True)

# MODEL
model, _ = load_m2f(CKPT_M2F, device)
if checkpoint is not None:
    model.load_state_dict(checkpoint['model_state_dict'])
    print("[OK] Model weights loaded")

# OPTIMIZER & SCHEDULER
optimizer = configure_optimizer(model, lr_backbone=1e-5, lr_head=5e-5, weight_decay=0.05)
scheduler = get_poly_scheduler(
    optimizer, num_epochs=10,
    steps_per_epoch=len(train_loader_wt),
    accum_steps=8, power=0.9, warmup_steps=200
)

if checkpoint is not None:
    try:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print("[OK] Optimizer state loaded")
    except Exception as e:
        print(f"[WARN] Could not load optimizer state: {e}")

    steps_to_catch_up = (start_epoch - 1) * (len(train_loader_wt) // 8)
    for _ in range(steps_to_catch_up):
        scheduler.step()
    print(f"[OK] Scheduler fast-forwarded {steps_to_catch_up} steps")

# LOSS dengan enhanced weights
loss_fn = FloodSegLoss(
    CLASS_WEIGHTS_ENHANCED.to(device),
    lovasz_weight=0.75,
    empty_classes=EMPTY_CLASSES
)

task3_config = {
    'batch_size': 2, 'accumulation_steps': 8, 'grad_clip': 1.0,
    'early_stop_patience': 5, 'num_epochs': 10,
    'save_dir': TRAIN_CONFIG['save_dir'],
}
trainer = Trainer(
    model=model, train_loader=train_loader_wt, val_loader=val_loader_wt,
    loss_fn=loss_fn, optimizer=optimizer, scheduler=scheduler,
    device=device, config=task3_config, use_amp=True,
    task_name='weighted'
)

if checkpoint is not None:
    trainer.best_score             = checkpoint.get('best_score', 0.0)
    trainer.best_smoothed_val_loss = checkpoint.get('best_smoothed_val_loss', float('inf'))
    trainer.patience_counter       = checkpoint.get('patience_counter', 0)
    trainer.history                = checkpoint.get('history', {
        'train_loss': [], 'val_loss': [], 'val_miou': [],
        'lr_backbone': [], 'lr_head': [], 'class_iou': []
    })

print(f"\nStarting weighted training from epoch {start_epoch}")
trainer.train(start_epoch=start_epoch)

torch.save({
    'epoch':                task3_config['num_epochs'],
    'model_state_dict':     model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_score':           trainer.best_score,
    'history':              trainer.history,
}, CKPT_M2F_WEIGHTED)
print(f"[OK] Saved → {CKPT_M2F_WEIGHTED}")

# SUBMISSION
df = generate_submission(
    model=model, test_img_dir=TEST_IMG, device=device,
    output_csv='submission_weighted.csv',
    use_tta=True, scales=[0.75, 1.0, 1.25], rotations=[0],
    class_thresholds=DEFAULT_CLASS_THRESHOLDS,
    model_type='m2f'
)
validate_submission('submission_weighted.csv')
print("[OK] Task 3 completed!")


In [ ]:
# ==================================================
# CELL 22: TASK 4 - Full TTA Submission
# ==================================================
print("="*60)
print("TASK 4: Full TTA (5 scales + 4 rotations + flip = 40 augs)")
print("="*60)

device = torch.device('cuda')
torch.cuda.empty_cache()

_candidate_ckpts = [
    CKPT_M2F_PSEUDO,      # path terbaru dari Cell 18 (iter terakhir)
    CKPT_M2F_WEIGHTED,
    CKPT_M2F_FINETUNE,
    CKPT_M2F,
]
best_ckpt = None
for ckpt_path in _candidate_ckpts:
    if ckpt_path and os.path.exists(ckpt_path):
        best_ckpt = ckpt_path
        break

if best_ckpt is None:
    raise FileNotFoundError(
        "No checkpoint found. Jalankan minimal Cell 15 (baseline training) dulu."
    )

model, ckpt = load_m2f(best_ckpt, device)
print(f"Using checkpoint: {best_ckpt}")
print(f"Best score      : {ckpt.get('best_score', ckpt.get('best_miou', 0)):.4f}")

# Full TTA: 5 scales x 4 rotations x 2 flip = 40 augmentations
df = generate_submission(
    model=model,
    test_img_dir=TEST_IMG,
    device=device,
    output_csv='submission_tta_rotation.csv',
    use_tta=True,
    scales=[0.75, 0.875, 1.0, 1.125, 1.25],
    rotations=[0, 90, 180, 270],
    class_thresholds=DEFAULT_CLASS_THRESHOLDS,
    model_type='m2f'
)

validate_submission('submission_tta_rotation.csv')
print("[OK] Task 4 completed!")


In [ ]:
# ==================================================
# CELL 23: TASK 6 - Automated Threshold Search & Final Submission
# ==================================================
print("="*60)
print("TASK 6: FINAL SUBMISSION (Automated Thresholds + Full TTA)")
print("="*60)

device = torch.device('cuda')
torch.cuda.empty_cache()

# 1. CARI CHECKPOINT TERBAIK
_candidate_ckpts = [
    CKPT_M2F_PSEUDO,
    CKPT_M2F_WEIGHTED,
    CKPT_M2F_FINETUNE,
    CKPT_M2F,
]
best_ckpt = None
for ckpt_path in _candidate_ckpts:
    if ckpt_path and os.path.exists(ckpt_path):
        best_ckpt = ckpt_path
        break

if best_ckpt is None:
    raise FileNotFoundError("No checkpoint found.")

model, ckpt = load_m2f(best_ckpt, device)
print(f"\nUsing checkpoint: {best_ckpt}")

# ==================================================
# [PHASE 1] CACHE VALIDATION PROBABILITIES
# ==================================================
print("\n[Phase 1] Caching validation probabilities for threshold search...")
val_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(VAL_IMG) if f.endswith('.jpg')])
val_ds_thresh = FloodSegDataset(VAL_IMG, VAL_MASK, transform=val_transform, image_ids=val_ids)
val_loader_thresh = DataLoader(val_ds_thresh, batch_size=2, shuffle=False, num_workers=2)

model.eval()
all_probs_val, all_masks_val = [], []
for images, masks, _ in tqdm(val_loader_thresh, desc="Caching", leave=False):
    images = images.to(device)
    with torch.no_grad():
        # Gunakan fungsi _model_to_probs dari Cell 13
        probs = _model_to_probs(model, images, 'm2f')
    all_probs_val.append(probs.cpu())
    all_masks_val.append(masks.cpu())

# ==================================================
# [PHASE 2] GREEDY THRESHOLD SEARCH
# ==================================================
print("\n[Phase 2] Greedy Threshold Search per Class...")
# Mulai dengan threshold netral (0.5 untuk semua kelas)
optimal_thresholds   = {c: 0.5 for c in range(NUM_CLASSES)}
# Batasi pencarian agar tidak overfit ke noise (0.20 - 0.80)
threshold_candidates = np.arange(0.20, 0.85, 0.05)

# Urutan prioritas optimasi: kelas langka (rare/water) diselesaikan lebih dulu
search_order = [9, 8, 4, 1, 3, 5, 7, 0]

for c in search_order:
    if c in EMPTY_CLASSES:
        continue

    best_iou = -1.0
    best_t   = optimal_thresholds[c]

    for t in threshold_candidates:
        temp_thresh = optimal_thresholds.copy()
        temp_thresh[c] = t

        metrics = FloodSegMetrics(NUM_CLASSES, EMPTY_CLASSES)
        for probs_batch, masks_batch in zip(all_probs_val, all_masks_val):
            preds = []
            for i in range(probs_batch.size(0)):
                pred = predict_with_per_class_threshold(probs_batch[i], temp_thresh)
                preds.append(pred)
            preds = np.stack(preds)
            metrics.update(preds, masks_batch.numpy())

        ious = metrics.compute_iou()
        class_iou = ious[c] if not np.isnan(ious[c]) else 0.0

        if class_iou > best_iou:
            best_iou = class_iou
            best_t   = t

    optimal_thresholds[c] = round(best_t, 2)
    print(f"  Class {c:<2} ({CLASS_NAMES[c][:15]:<15}) -> Opt Thresh: {optimal_thresholds[c]:.2f} (Val IoU: {best_iou:.4f})")

print(f"\nFinal Optimized Thresholds:\n{optimal_thresholds}")

# ==================================================
# [PHASE 3] FINAL INFERENCE
# ==================================================
print("\n[Phase 3] Generating Final Submission with Full TTA & Optimal Thresholds...")
df = generate_submission(
    model=model,
    test_img_dir=TEST_IMG,
    device=device,
    output_csv='submission_final.csv',
    use_tta=True,
    scales=[0.75, 0.875, 1.0, 1.125, 1.25],
    rotations=[0, 90, 180, 270],
    class_thresholds=optimal_thresholds,
    model_type='m2f'
)

validate_submission('submission_final.csv')
print("\n[OK] TASK 6 - FINAL SUBMISSION READY!")


In [ ]:
# ==================================================
# CELL 24: Download All Submissions
# ==================================================
from google.colab import files

submission_files = [
    'submission_baseline.csv',
    'submission_ensemble.csv',
    'submission_finetune.csv',
    'submission_weighted.csv',
    'submission_tta_rotation.csv',
    'submission_final.csv',
]

print("="*60)
print("DOWNLOADING ALL SUBMISSIONS")
print("="*60)

for fname in submission_files:
    if os.path.exists(fname):
        size_kb = os.path.getsize(fname) / 1024
        print(f"Downloading {fname} ({size_kb:.1f} KB)...")
        files.download(fname)
    else:
        print(f"[WARN] {fname} not found, skip")

print("\n[OK] Done!")
